# Notebook 6 — Improving Copilot Reliability & Final Review

### Build an AI Business Copilot 🏔️ · Session 2

You've built a complete copilot: it routes any question to the right skill and answers with
grounded facts. But **"it worked on the questions I tried"** is not the same as **"I trust
it with customers."** This final notebook is about earning that trust.

By the end you'll be able to:

1. Build a small **evaluation set** and score the copilot automatically.
2. Read the results to find where it slips.
3. **Iterate** — change one thing, re-measure, and see if it helped.
4. Apply practical **cost controls**.
5. Leave with a concrete plan to adapt this copilot to **your own** use case.

> 💸 This notebook runs the copilot over ~8 evaluation questions — still Haiku 4.5, a few
> cents at most.

## 1. Setup

Same setup as Session 2. Run the setup cells, then the "complete copilot" cell (it's the
code you already wrote across Notebooks 2–5, packaged so we can focus on evaluating it).

In [ ]:
%pip install -q anthropic sentence-transformers faiss-cpu pandas numpy

In [ ]:
# --- Make the workshop files available (Colab, Databricks, or local) ---
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/YOUR-ORG/trailhead-copilot"   # <-- ✏️ set to the real repo URL

def find_repo_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if all((base / d).is_dir() for d in ("data", "documents", "src")):
            return base
    cloned = Path.cwd() / "trailhead-copilot"
    return cloned if (cloned / "src").is_dir() else None

root = find_repo_root()
if root is None:
    print("Cloning the workshop repo...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    root = find_repo_root()
if root is None:
    raise RuntimeError("Workshop files not found — set REPO_URL to the real repository URL.")

sys.path.insert(0, str(root / "src"))
print("Using workshop files at:", root)

In [ ]:
import os, getpass
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Paste your Anthropic API key: ")
print("API key is set. ✅")

In [ ]:
MODEL = "claude-haiku-4-5"
MAX_TOKENS = 500
TOP_K = 4

## 2. The complete copilot (review)

This one cell reassembles everything from Notebooks 2–5 — retriever, document skill, data
tools, router, and `ask_copilot()`. You've seen every line before; run it and move on.

In [ ]:
import trailhead as th
import anthropic, json
import pandas as pd

# --- Skill 1: documents (Notebooks 2-3) ---
chunks = th.build_chunks()
index, embedder = th.build_index(chunks)

def retrieve(query, k=None):
    return th.search(query, index, chunks, embedder, top_k=k or TOP_K)

SYSTEM_DOCS = (
    "You are a support assistant for Trailhead Supply Co. Answer using ONLY the policy "
    "excerpts provided. If the answer isn't in them, say you don't have that information. "
    "Keep it short and cite the document like [source: return_policy.md]."
)

def build_context(hits):
    return "\n\n".join("[source: " + h["source"] + "]\n" + h["text"] for h in hits)

def answer_from_documents(query):
    prompt = "POLICY EXCERPTS:\n" + build_context(retrieve(query)) + "\n\nCUSTOMER QUESTION: " + query
    return th.ask_claude(prompt, system=SYSTEM_DOCS, model=MODEL, max_tokens=MAX_TOKENS)

# --- Skill 2: data tools (Notebook 4) ---
data = th.load_data()
orders, shipments = data["orders"], data["shipments"]
products, inventory = data["products"], data["inventory"]
client = anthropic.Anthropic()

def clean(v):
    return None if pd.isna(v) else v

def get_order_status(order_id):
    order_id = int(order_id)
    o = orders[orders["order_id"] == order_id]
    if o.empty:
        return {"error": "No order found with ID " + str(order_id)}
    o = o.iloc[0]
    r = {"order_id": order_id, "order_status": clean(o["status"]),
         "shipping_method": clean(o["shipping_method"]), "order_total": float(o["total"])}
    s = shipments[shipments["order_id"] == order_id]
    if not s.empty:
        s = s.iloc[0]
        for k in ["carrier", "tracking_number", "estimated_delivery", "delivered_date"]:
            r[k] = clean(s[k])
        r["shipment_status"] = clean(s["status"])
    return r

def check_inventory(product_name):
    m = products[products["name"].str.contains(product_name, case=False, na=False)]
    if m.empty:
        return {"error": "No product matching '" + product_name + "'."}
    found = []
    for _, p in m.iterrows():
        inv = inventory[inventory["product_id"] == p["product_id"]]
        units = int(inv.iloc[0]["units_in_stock"]) if not inv.empty else 0
        found.append({"product": p["name"], "units_in_stock": units, "in_stock": units > 0})
    return {"matches": found}

tools = [
    {"name": "get_order_status",
     "description": "Look up status, shipping, carrier, tracking, estimated delivery for an order by numeric ID.",
     "input_schema": {"type": "object", "properties": {"order_id": {"type": "integer", "description": "Numeric order ID"}}, "required": ["order_id"]}},
    {"name": "check_inventory",
     "description": "Check units in stock for a product by full or partial name.",
     "input_schema": {"type": "object", "properties": {"product_name": {"type": "string", "description": "Full or partial product name"}}, "required": ["product_name"]}},
]
TOOL_FUNCTIONS = {"get_order_status": get_order_status, "check_inventory": check_inventory}
SYSTEM_DATA = ("You are a Trailhead Supply Co. support assistant. Use the tools to look up "
               "real order and inventory data. Only state facts returned by a tool. Keep replies brief.")

def run_with_tools(user_message, system):
    messages = [{"role": "user", "content": user_message}]
    while True:
        resp = client.messages.create(model=MODEL, max_tokens=MAX_TOKENS, system=system, tools=tools, messages=messages)
        if resp.stop_reason != "tool_use":
            return "".join(b.text for b in resp.content if b.type == "text")
        messages.append({"role": "assistant", "content": resp.content})
        results = []
        for block in resp.content:
            if block.type == "tool_use":
                out = TOOL_FUNCTIONS[block.name](**block.input)
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(out)})
        messages.append({"role": "user", "content": results})

def answer_from_data(query):
    return run_with_tools(query, SYSTEM_DATA)

# --- Router + one door (Notebook 5) ---
ROUTER_SYSTEM = (
    "Classify the customer question for a support copilot. Reply with ONLY one word:\n"
    "- policy: return/shipping/warranty rules, membership, or gear care.\n"
    "- data: a specific order, shipment, or product stock level.\n"
    "- both: needs a policy rule AND a specific order/product lookup."
)

def route(question):
    label = th.ask_claude(question, system=ROUTER_SYSTEM, model=MODEL, max_tokens=5).strip().lower()
    for category in ("both", "policy", "data"):
        if category in label:
            return category
    return "both"

SYSTEM_BOTH = ("You are a Trailhead Supply Co. support assistant. Use the policy excerpts in "
               "the message for rules and the tools for the specific order/product. Combine both.")

def answer_both(question):
    prompt = "POLICY EXCERPTS:\n" + build_context(retrieve(question)) + "\n\nCUSTOMER QUESTION: " + question
    return run_with_tools(prompt, SYSTEM_BOTH)

def ask_copilot(question, verbose=True, category=None):
    if category is None:
        category = route(question)
    if verbose:
        print(f"[router -> {category}]")
    if category == "policy":
        return answer_from_documents(question)
    if category == "data":
        return answer_from_data(question)
    return answer_both(question)

print("Copilot ready. Quick check:")
print(ask_copilot("What is the return window for boots?"))

## 3. Why evaluate?

When you change a prompt, add a document, or swap a model, how do you know you made things
**better** and not worse? You can't eyeball a handful of questions and be sure. The
professional answer is an **evaluation set**: a fixed list of questions with a way to check
each answer. Run it before and after any change, and you get a **score** you can trust.

We'll keep it lightweight and transparent — a hand-built set plus simple checks — so you can
see exactly how it works and scale it up later.

## 4. Build an evaluation set

Each item has a **question**, the route we **expect**, and a way to check the answer:

- `expect` — words the answer *should* contain (any one counts), or
- `decline` — the copilot *should* say it doesn't have the info.

In [ ]:
EVAL_SET = [
    {"q": "What is the return window for hiking boots?", "route": "policy", "expect": ["30"]},
    {"q": "Can I return a final sale item?", "route": "policy",
     "expect": ["final sale", "not returnable", "cannot", "can't", "non-returnable"]},
    {"q": "What does the warranty cover on a tent?", "route": "policy",
     "expect": ["defect", "2 year", "2-year", "manufactur"]},
    {"q": "Where is order 1001?", "route": "data", "expect": ["transit", "shipped", "ups", "1001"]},
    {"q": "Is the Voyager 65L Backpack in stock?", "route": "data", "expect": ["stock", "70", "available"]},
    {"q": "How long do I have to return order 1007?", "route": "both", "expect": ["30"]},
    {"q": "Do you sell gift cards?", "route": None, "decline": True},
    {"q": "What is the status of order 9999?", "route": "data", "decline": True},
]
print("Evaluation set:", len(EVAL_SET), "questions")

## 5. Score the copilot automatically

Our checker runs each question through `ask_copilot()` and marks it **pass** only if the
route is right *and* the answer passes its content check. Simple keyword checks like this
are surprisingly effective for a first evaluation — and completely transparent.

In [ ]:
DECLINE_PHRASES = [
    "don't have", "do not have", "not sure", "no order", "couldn't find", "could not find",
    "not found", "no record", "unable", "no information", "contact support", "don't sell", "do not sell",
]

def contains_any(text, needles):
    t = text.lower()
    return any(n.lower() in t for n in needles)

def evaluate(eval_set, verbose_fail=True):
    rows, passed = [], 0
    for item in eval_set:
        category = route(item["q"])
        answer = ask_copilot(item["q"], verbose=False, category=category)
        route_ok = (item.get("route") is None) or (category == item["route"])
        if item.get("decline"):
            content_ok = contains_any(answer, DECLINE_PHRASES)
        else:
            content_ok = contains_any(answer, item["expect"])
        ok = route_ok and content_ok
        passed += int(ok)
        rows.append({"q": item["q"], "route": category, "ok": ok, "answer": answer})
    print(f"SCORE: {passed}/{len(eval_set)} passed\n")
    for x in rows:
        mark = "✅" if x["ok"] else "❌"
        print(f"{mark} [{x['route']:6}] {x['q']}")
        if not x["ok"] and verbose_fail:
            print("      -> " + x["answer"][:140].replace(chr(10), " "))
    return rows

results = evaluate(EVAL_SET)

## 6. A second opinion — LLM-as-judge

Keyword checks can't tell if an answer is *well-written* or subtly wrong. A complementary
technique is **LLM-as-judge**: ask a second Claude call to grade the answer. It's more
flexible than keywords (and more like how a human would grade), though it costs a call and
isn't perfect. Use both: keywords for cheap, deterministic checks; a judge for nuance.

In [ ]:
JUDGE_SYSTEM = (
    "You grade a customer-support answer. Given a QUESTION and the copilot's ANSWER, reply "
    "with PASS or FAIL, then a short reason. PASS if the answer is helpful and invents no "
    "facts; FAIL if it is wrong, invented, or unhelpful."
)

def judge(question, answer):
    return th.ask_claude("QUESTION: " + question + "\n\nANSWER: " + answer,
                         system=JUDGE_SYSTEM, model=MODEL, max_tokens=60)

for q in ["What is the return window for hiking boots?", "Do you sell gift cards?"]:
    a = ask_copilot(q, verbose=False)
    print("Q:", q)
    print("A:", a)
    print("JUDGE:", judge(q, a))
    print("-" * 70)

## 7. Iterate — change one thing, re-measure

This is the real development loop: **measure → change one thing → measure again.** Let's try
tightening the document instructions to be more explicit, then re-run the evaluation and
compare the score. In your own project you'd try each candidate change this way and keep
only the ones that improve the score.

In [ ]:
# Save the baseline, then try a change (a stricter document system prompt).
baseline_score = sum(r["ok"] for r in results)

SYSTEM_DOCS = (
    "You are a Trailhead Supply Co. support assistant. Answer ONLY from the policy excerpts. "
    "State the specific number (days, months, dollars) when the policy gives one. "
    "If the answer isn't in the excerpts, say you don't have that information. "
    "Cite the document like [source: return_policy.md]."
)

print("Re-running evaluation after the change...\n")
new_results = evaluate(EVAL_SET, verbose_fail=False)
print(f"\nBaseline: {baseline_score}/{len(EVAL_SET)}   ->   After change: {sum(r['ok'] for r in new_results)}/{len(EVAL_SET)}")

> ✏️ **Try it.** Pick another lever — raise `TOP_K` to `6`, or lower it to `2` — re-run the
> config cell and `evaluate(EVAL_SET)`, and see how the score moves. Small, measured changes
> beat guessing. (With only 8 questions the score may not move much; that's why real teams
> use dozens or hundreds of eval questions.)

## 8. Reliability & guardrails

A few habits that make a copilot safer to ship:

- **Always give the "I don't know" escape.** Every answering prompt tells Claude to decline
  when the facts aren't there — this is your main defense against confident wrong answers.
- **Keep answers grounded and cited.** Citations let a human verify; grounding keeps the
  model from drifting into invention.
- **Watch the router.** A mis-route sends a question to the wrong skill. Your eval set is how
  you *catch* mis-routes; when you find one, add it to the set so you never regress on it.
- **Fail gracefully.** Your tools return clear errors on a miss, and the model relays them —
  no crashes, no fabrication.
- **Keep a human in the loop.** For refunds, complaints, or anything high-stakes, a copilot
  should hand off to a person rather than act on its own.

## 9. Cost controls

Practical AI is cost-aware. The main levers:

| Lever | Effect |
| --- | --- |
| **Model choice** | Haiku 4.5 (~$1 / $5 per million in/out tokens) is far cheaper than Sonnet or Opus, and plenty for this workload. Reach for a bigger model only where you can *measure* it's needed. |
| **`max_tokens`** | Caps the reply length — you pay for output tokens. Don't set it higher than answers need. |
| **`TOP_K`** | Fewer retrieved chunks = less context sent = cheaper (and often just as accurate). |
| **Prompt caching** | Reuse a large, unchanging context across calls at ~10% of the price (see below). |

You can estimate the input cost of any prompt *before* sending it, with `count_tokens`:

In [ ]:
sample_prompt = "POLICY EXCERPTS:\n" + build_context(retrieve("return window")) + "\n\nCUSTOMER QUESTION: How long to return boots?"
count = client.messages.count_tokens(
    model=MODEL, system=SYSTEM_DOCS,
    messages=[{"role": "user", "content": sample_prompt}],
)
print("Input tokens for one document answer:", count.input_tokens)
print("Approx input cost at Haiku pricing: $", round(count.input_tokens * 1e-6, 6))

### Prompt caching (when your shared context is large)

If many requests share a big, unchanging chunk of text — say you put *all* of Trailhead's
policies in every prompt — you can **cache** that prefix so you're not re-paying to process
it each time. You mark the stable part with `cache_control`, and repeat calls read it at a
fraction of the cost:

```python
# Illustrative — caching pays off for LARGE shared prefixes.
client.messages.create(
    model=MODEL, max_tokens=MAX_TOKENS,
    system=[{"type": "text", "text": VERY_LARGE_SHARED_CONTEXT,
             "cache_control": {"type": "ephemeral"}}],
    messages=[{"role": "user", "content": question}],
)
```

> ⚠️ Caching only kicks in above a model-specific minimum (a few thousand tokens), so it
> helps when you have a **large** shared context — not for our short prompts here. Good to
> know it exists for when your copilot grows.

## 10. Final review — and making it your own

Look how far you came. Every box in the Notebook 1 architecture is now real and evaluated:

```
Question -> ROUTER -> [ documents (RAG) | data (tools) | both ] -> grounded, cited answer
```

You built a practical business copilot combining **LLM prompting, RAG, structured data,
workflow routing, and evaluation** — and you did it all by **grounding** a model, never
training one.

### 🎒 Your turn: adapt this template to your own fictional company

The workshop project is a template. To make it yours:

1. **Swap the documents.** Replace the files in `documents/markdown/` with your own policies
   or knowledge, and rebuild the index.
2. **Swap the data.** Replace the CSVs in `data/` with your own tables (customers, orders,
   whatever fits your business).
3. **Adjust the tools.** Change the lookup functions and their tool descriptions to match
   your data's columns.
4. **Tune the router.** Update the categories to your kinds of questions.
5. **Write an eval set** for *your* copilot and use it to guide changes.

That's the whole recipe for a practical GenAI business application.

## 11. Recap

- **Evaluate, don't eyeball.** A fixed question set with checks gives a score you can trust.
- **Keyword checks** are cheap and transparent; **LLM-as-judge** adds nuance.
- **Iterate** by changing one thing and re-measuring.
- **Guardrails** ("I don't know," citations, graceful failure, human handoff) make it safe.
- **Cost** is a design choice: model, `max_tokens`, `TOP_K`, and caching.
- The copilot is a **template** — swap the docs, data, tools, and router to fit any business.

## 12. Exercises

**Guided**
1. Add two new questions to `EVAL_SET` — one policy, one data — with sensible checks, and
   re-run `evaluate()`.
2. Find a question the copilot gets wrong (tweak wording until you do). Add it to the eval
   set so any future change is measured against it.

**Challenge**
3. Turn the two scores in Section 7 into a proper A/B test: wrap the whole "set a prompt →
   evaluate" flow in a function, try three different `SYSTEM_DOCS` wordings, and print a
   leaderboard.
4. Combine keyword and judge scoring: extend `evaluate()` to also call `judge()` on each
   answer and report both a keyword score and a judge score. When do they disagree?

## 13. Learn more

- **Anthropic — Create strong empirical evaluations:** <https://docs.claude.com/en/docs/test-and-evaluate/develop-tests>
- **Anthropic — Reduce hallucinations (guardrails):** <https://docs.claude.com/en/docs/test-and-evaluate/strengthen-guardrails/reduce-hallucinations>
- **Anthropic — Prompt caching:** <https://docs.claude.com/en/docs/build-with-claude/prompt-caching>
- **Anthropic — Pricing:** <https://www.anthropic.com/pricing>
- **Codecademy — Intro to Generative AI:** <https://www.codecademy.com/learn/intro-to-generative-ai>

---
> ### 👩‍🏫 Instructor notes (remove before distributing to students)
>
> - **Timing (~55 min):** ~8 setup + rebuild, ~15 eval set + scoring (§3–5, the core),
>   ~8 LLM-judge (§6), ~8 iterate (§7), ~8 cost (§9), ~8 final review + extension (§10).
> - **The eval score is the payoff of the whole workshop** — frame evaluation as what
>   separates a demo from something you'd ship. Expect roughly 7–8/8 on this set; if one
>   fails (often a router edge case), that's a *great* live teaching moment for §7.
> - §7 iterate: with only 8 questions the score may not move — say so, and stress that real
>   teams use dozens/hundreds. The **method** (change one thing, re-measure) is the lesson.
> - **Prompt caching (§9) is deliberately conceptual** — our prompts are below the cache
>   minimum, so a live demo would show zero cache hits. Explain rather than run it.
> - §10 is the send-off: make sure everyone leaves knowing the 5-step recipe to adapt the
>   template to their own fictional company (that's their post-workshop project).
> - **Cost:** an eval run is ~8 questions x (1 router + 1-2 skill calls) plus a couple of
>   judge calls — a few cents per student at most.